# 01 Cleaning
Goal: load the raw inspections file, clean it, and save a processed dataset plus a one-row-per-violation table.

Write a short note under each step about what you found.

In [ ]:
import pandas as pd
import numpy as np

raw_path = "../data/raw/food_inspections.csv"
df = pd.read_csv(raw_path)
df.shape

## Step 1: Inspect
Look at columns, types, missing values, and duplicates.

In [ ]:
df.info()
df.head()

In [ ]:
df.isna().sum().sort_values(ascending=False)

In [ ]:
df.duplicated().sum()

## Step 2: Standardize column names

In [ ]:
df.columns = (df.columns.str.strip().str.lower()
              .str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_"))
df.columns.tolist()

## Step 3: Dates and text
Convert the date, tidy text columns, and look at the `results` values.

In [ ]:
df["inspection_date"] = pd.to_datetime(df["inspection_date"], errors="coerce")
for col in ["facility_type", "risk", "results", "dba_name"]:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip().str.title()
df["results"].value_counts()

## Step 4: Create analysis columns
Decide what counts as a fail. Here only the result 'Fail' does; write down your choice and why.

In [ ]:
df["is_fail"] = df["results"].eq("Fail")
df["inspection_year"] = df["inspection_date"].dt.year
df["inspection_month"] = df["inspection_date"].dt.month
df[["results", "is_fail", "inspection_year"]].head()

## Step 5: Split the violations text
Each inspection has many violations in one cell, separated by `|`. Make one row per violation.

In [ ]:
viol = df[["inspection_id", "violations"]].dropna().copy()
viol["violations"] = viol["violations"].str.split("|")
viol = viol.explode("violations")
viol["violations"] = viol["violations"].str.strip()

# Pull out the violation number and description; drop the inspector's comments
extracted = viol["violations"].str.extract(r"^(\d+)\.\s*(.*?)(?:\s*-\s*Comments:.*)?$", expand=True)
viol["violation_code"] = pd.to_numeric(extracted[0], errors="coerce")
viol["violation_desc"] = extracted[1].str.title()
viol = viol.drop(columns="violations").dropna(subset=["violation_code"])
viol.head()

## Step 6: Check and save
Check row counts and a few random rows before saving.

In [ ]:
keep = ["inspection_id", "dba_name", "license", "facility_type", "risk", "zip",
        "inspection_date", "inspection_type", "results", "is_fail",
        "latitude", "longitude"]
keep = [c for c in keep if c in df.columns]
inspections = df[keep].drop_duplicates("inspection_id").rename(columns={"license": "license_no"})

inspections.to_csv("../data/processed/inspections_clean.csv", index=False)
viol.to_csv("../data/processed/violations_clean.csv", index=False)
inspections.shape, viol.shape

## What I learned / data issues I found
- 